### After clearing gates:
1. Data Governance: Must data stay private & never leave machine?
    - self hosted/No API
2. Cost Cieling: Zero budget -> every paid API is out.
3. hardware fit --> Does it actually run on my device?
    - Too big to load = disqualified, not "low quality"

The selected model needs to be scored based on:
1. faithfulness
2. Grounding
3. Refusal behaviour
4. Citation / tracibility
5. Controllability and latency


### Selected models

1. Llama 3.2 3B Instruct	3B	128K	Meta Llama Community License
2. Qwen2.5-3B-Instruct	3B	32K	Qwen custom license
3. Microsoft Phi-4-mini-instruct	3.8B	128K	MIT	Strongest small option for reasoning and document QA
4. Ministral 3B Instruct	3B	128K	Mistral Research/custom license
5. IBM Granite 3.3 2B Instruct	2B	128K	Apache 2.0	Good long-context and enterprise-document option

> The above model still needs to be calculate dfor the how much memory they will occupy during runtime and overhead because of kv caching


## # Creating a function to calculate the KV cache size overhead size and the estimate of the model size that fir in the model or not

In [97]:
## I need some component details of the architecture to calculate th =e kv cache size
# Number of kv heads
# Number of layers
# The output dimenion of the model
import requests
from huggingface_hub import ModelCard
def pull_config(model_id):
    url = f"https://huggingface.co/{model_id}/resolve/main/config.json"
    content = requests.get(url).json()
    mechanics = {"model": model_id.split("/")[1],
                 "context_window": content.get('max_position_embeddings',""),
                 "precision": content.get('torch_dtype',""),
                 "specs": {"num_layers": content.get("num_hidden_layers",""),
                           "num_kv_heads": content.get("num_key_value_heads",""),
                           "num_attention_heads": content.get("num_attention_heads",""),
                           "head_size": content.get("hidden_size","")}
                           }
    return mechanics

pull_config("meta-llama/Llama-3.2-3B-Instruct")

    

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [86]:
def calculate_kv_cache_size(model_id, token_sequence_length=8000):
    model_specs = pull_config(model_id)
    if model_specs['precision'] == 'bfloat16':
        bytes_per_number = 2
    elif model_specs['precision'] == 'F32' or model_specs['precision'] == 'FP32':
        bytes_per_number = 4
    elif model_specs['precision'] == 'F8' or model_specs['precision'] == 'FP8':
        bytes_per_number = 1
    else:
        raise ValueError(f"Unknown precision: {model_specs['precision']}")

    num_kv_heads = model_specs['specs']['num_kv_heads']
    num_layers = model_specs['specs']['num_layers']
    head_dim = model_specs['specs']['head_size']/model_specs['specs']['num_attention_heads']

    peak_kv = 2*num_kv_heads*num_layers*head_dim*bytes_per_number*token_sequence_length

    kv_in_MB = peak_kv / (1024*1024)

    return f"{kv_in_MB} MB", model_specs

In [87]:
calculate_kv_cache_size("ibm-granite/granite-3.3-2b-instruct")

('625.0 MB',
 {'model': 'granite-3.3-2b-instruct',
  'context_window': 131072,
  'precision': 'bfloat16',
  'specs': {'num_layers': 40,
   'num_kv_heads': 8,
   'num_attention_heads': 32,
   'head_size': 2048}})

In [54]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('BAAI/bge-base-en-v1.5')


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2818.36it/s]
BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [63]:
total_params = sum(p.numel() for p in model.parameters())

In [61]:
# check precision
for p in model.parameters():
    print(p.dtype)
    break

torch.float32


### Now we know the embedding model has 110M params and FP32 precision

In [65]:
# Calculate the size of embedding model
byte_per_numner = 4
embedder_size = (total_params * 4)/(1024*1024)
print(f"{embedder_size} MB")

417.6416015625 MB


### Create a estimator function
* I am going to use ollama and Q4 quantised model which is default so the weights precision will be `4-bit` and byte per number will be `0.5`

In [88]:
def estimate_memory_usage(model_id, model_size_from_ollama, embedder_size = 417.6416, overhead = 1000):
    kv_cache_size, model_spec = calculate_kv_cache_size(model_id)
    cache_size = float(kv_cache_size.split()[0])/1024
    
    Total_estimated_memory_usage = cache_size + embedder_size/1024 + model_size_from_ollama + overhead/1024

    total_usage_in_GB = Total_estimated_memory_usage
    return f"{total_usage_in_GB} GB", model_spec



In [95]:
granite33_2b = estimate_memory_usage("ibm-granite/granite-3.3-2b-instruct",model_size_from_ollama=1.5)
qwen25_3b = estimate_memory_usage("Qwen/Qwen2.5-3B-Instruct", model_size_from_ollama=1.9)
phi4 = estimate_memory_usage("microsoft/Phi-4-mini-instruct",model_size_from_ollama=2.5)
#llama = estimate_memory_usage("meta-llama/Llama-3.2-3B-Instruct", model_size_from_ollama=2)
ministral = estimate_memory_usage("mistralai/Ministral-3-3B-Instruct-2512", model_size_from_ollama=3)

ValueError: Unknown precision: 

In [96]:
granite33_2b, qwen25_3b,phi4,llama, ministral

NameError: name 'llama' is not defined

In [107]:
headers = {"Authorization": f"Bearer HF_TOKEN_REMOVED"}
url = "https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct/resolve/main/config.json"
content = requests.get(url, headers=headers)

In [109]:
content.content

b'Access to model meta-llama/Llama-3.2-3B-Instruct is restricted. You must have access to it and be authenticated to access it. Please log in.'